# Do Premier League wages track performance?

**Question:** Given how well a player performed in 2025-26 for his role, what *should* he be paid, and how far is that from his real wage?

- **Performance:** EPL 2025-26 player stats (up to GameDay 35)
- **Roles:** Transfermarkt detailed positions (`sub_position`)
- **Wages:** Player salary estimates (2024-25)
- Only players with 900+ minutes who are **at the same club in both files** are compared, so old wages from a previous club don't skew the results.

In [1]:
import html, unicodedata, difflib
import numpy as np
import pandas as pd

PERF_PATH = "data/raw/player_stats_2025_26.csv"
WAGE_PATH = "data/raw/player_salaries.csv"
ROLE_PATH = "data/raw/player_roles_transfermarkt.csv"
OUT_DIR   = "data/output"


## 1. Load both files and make names comparable

In [2]:
def clean_name(n):
    n = str(n)
    for a, b in {"Đ": "D", "đ": "d", "Ø": "O", "ø": "o", "ß": "ss"}.items():
        n = n.replace(a, b)
    n = unicodedata.normalize("NFKD", n).encode("ascii", "ignore").decode()
    return n.lower().replace("-", " ").strip()

# The two files spell some clubs differently
club_map = {
    "Manchester Utd": "Manchester United", "Newcastle Utd": "Newcastle United",
    "Nott'ham Forest": "Nottingham Forest", "Tottenham": "Tottenham Hotspur",
    "West Ham": "West Ham United", "Wolves": "Wolverhampton",
    "Brighton & Hove Albion": "Brighton", "Liverpool FC": "Liverpool",
}
def clean_club(t):
    t = html.unescape(str(t))          # fixes "Brighton &amp; Hove Albion"
    return club_map.get(t, t)

perf = pd.read_csv(PERF_PATH).copy()
perf["team_name"] = perf["team_name"].apply(html.unescape)
perf["club"] = perf["team_name"].apply(clean_club)
perf["key"] = perf["player_name"].apply(clean_name)

wages = pd.read_csv(WAGE_PATH)
wages["club"] = wages["Team"].apply(clean_club)
wages["key"] = wages["Player"].apply(clean_name)
wages = wages.rename(columns={"Annual": "annual_wage", "Age": "age_2024_25"})

print("performance rows:", len(perf), "| wage rows:", len(wages))

performance rows: 526 | wage rows: 684


## 2. Keep regular players and match them to a wage **at the same club**

Matching only within the same club means a renamed or fuzzy match can't accidentally grab someone at another team, and players who changed clubs (stale wages) drop out automatically.

In [3]:
perf = perf[perf["minutesPlayed"] >= 900].copy()

def find_wage_row(r):
    pool = wages[wages["club"] == r["club"]]
    if pool.empty:
        return None
    keys = pool["key"].tolist()
    if r["key"] in keys:                                         # exact name
        return pool.index[keys.index(r["key"])]
    close = difflib.get_close_matches(r["key"], keys, n=1, cutoff=0.8)
    if close:                                                    # small spelling difference
        return pool.index[keys.index(close[0])]
    last = r["key"].split()[-1]                                  # same surname, only one at the club
    same_last = [i for i, k in zip(pool.index, keys) if k.split()[-1] == last]
    return same_last[0] if len(same_last) == 1 else None

perf["wage_row"] = perf.apply(find_wage_row, axis=1)
w = perf[perf["wage_row"].notna()].copy()
w = w.join(wages.loc[w["wage_row"].astype(int), ["Player", "annual_wage", "age_2024_25"]]
           .set_index(w.index))

print(f"{len(perf)} players with 900+ minutes")
print(f"{len(w)} matched to a wage at the same club")

# Check the non-exact matches look right
odd = w[w["key"] != w["Player"].apply(clean_name)][["player_name", "Player", "team_name"]]
print(f"\nNon-exact name matches ({len(odd)}):")
odd

326 players with 900+ minutes
203 matched to a wage at the same club

Non-exact name matches (7):


,player_name,Player,team_name
36,Nico González,Nicolás González,Manchester City
91,Andy Robertson,Andrew Robertson,Liverpool FC
110,Emiliano Buendía,Emi Buendía,Aston Villa
166,Yehor Yarmoliuk,Yegor Yarmolyuk,Brentford
185,Ferdi Kadıoğlu,Ferdi Kadioglu,Brighton & Hove Albion
274,Joshua King,Josh King,Fulham
316,Tino Livramento,Valentino Livramento,Newcastle United


## 3. Detailed roles and the stats that matter for each

The stats file only says G / D / M / F, which lumps wingers in with holding midfielders and full-backs in with centre-backs. Transfermarkt's `sub_position` (`data/raw/player_roles_transfermarkt.csv`) splits players into **6 roles**, so each player is judged only on what his role is actually asked to do.

| Role | Focus | Built from (closest stats in our data) |
|---|---|---|
| Goalkeeper | Shot stopping, crosses, distribution | goals prevented, saves, high claims, pass accuracy |
| Centre-back | Defending, aerial duels, ball progression | interceptions, recoveries, aerial win %, tackles won %, long balls |
| Full-back | Width, crossing, 1v1 defending | xA, key passes, crosses, ground duels, tackles |
| Central / defensive mid | Control and ball winning | pass %, passes into opposition half, tackles + interceptions, recoveries |
| Winger / attacking mid | Creating and beating players | xA, key passes, big chances created, dribbles, goals |
| Striker | Finishing and box presence | goals, xG, goals above xG, conversion %, shots on target % |

Each stat has a **weight**: the role's core stats count 2-3x, supporting stats count 1x. Stats like PSxG, progressive carries, xThreat and pressing aren't in this dataset, so the closest available stat is used instead.

In [4]:
# Map Transfermarkt sub_position -> role (fallback uses the G/D/M/F from the stats file)

role_map = {
    "Goalkeeper": "Goalkeeper",
    "Centre-Back": "Centre-back",
    "Left-Back": "Full-back", "Right-Back": "Full-back",
    "Defensive Midfield": "Central / defensive mid", "Central Midfield": "Central / defensive mid",
    "Attacking Midfield": "Winger / attacking mid", "Left Winger": "Winger / attacking mid",
    "Right Winger": "Winger / attacking mid", "Left Midfield": "Winger / attacking mid",
    "Right Midfield": "Winger / attacking mid",
    "Centre-Forward": "Striker", "Second Striker": "Striker",
}
fallback = {"G": "Goalkeeper", "D": "Centre-back", "M": "Central / defensive mid", "F": "Striker"}

roles = (pd.read_csv(ROLE_PATH, usecols=["player_name", "sub_position"])
           .drop_duplicates("player_name"))
perf = perf.drop(columns=["sub_position"], errors="ignore").merge(roles, on="player_name", how="left") \
           .set_index(perf.index)
perf["role"] = perf["sub_position"].map(role_map).fillna(perf["position"].map(fallback))

# Fix: Transfermarkt's file matched the wrong (younger) Idrissa Gueye, who is a striker
perf.loc[perf["player_name"] == "Idrissa Gana Gueye", "role"] = "Central / defensive mid"

# Players whose 2025-26 role differs from Transfermarkt's main position
role_overrides = {
    "Bernardo Silva": "Central / defensive mid",
}
for player, role in role_overrides.items():
    perf.loc[perf["player_name"] == player, "role"] = role

w["role"] = perf.loc[w.index, "role"]

print("players with a detailed role from Transfermarkt:", perf["sub_position"].notna().sum(), "of", len(perf))
print(perf["role"].value_counts().to_string())

# Extra stats built from the raw columns
perf["tacklesPlusInterceptions"] = perf["tackles"].fillna(0) + perf["interceptions"].fillna(0)
perf["shotsOnTargetPercentage"] = (perf["shotsOnTarget"] / perf["totalShots"].replace(0, np.nan) * 100).fillna(0)
perf["possessionLostPercentage"] = (perf["possessionLost"] / perf["touches"].replace(0, np.nan) * 100).fillna(0)

# Non-penalty versions, so penalty takers don't get free goals and xG (a penalty is worth ~0.76 xG)
perf["nonPenaltyGoals"] = perf["goals"].fillna(0) - perf["penaltyGoals"].fillna(0)
perf["nonPenaltyXG"] = perf["expectedGoals"].fillna(0) - 0.76 * perf["penaltiesTaken"].fillna(0)
perf["goalsAboveXG"] = perf["nonPenaltyGoals"] - perf["nonPenaltyXG"]

# role -> {stat: (higher_is_better, weight)}
role_stats = {
    "Goalkeeper": {
        "goalsPrevented": (True, 3), "saves": (True, 2), "savedShotsFromInsideTheBox": (True, 2),
        "highClaims": (True, 1.5), "punches": (True, 1), "successfulRunsOut": (True, 1),
        "accuratePassesPercentage": (True, 1.5), "accurateLongBallsPercentage": (True, 1.5),
        "cleanSheet": (True, 1), "errorLeadToGoal": (False, 2), "crossesNotClaimed": (False, 1),
    },
    "Centre-back": {
        # Defending
        "interceptions": (True, 2), "ballRecovery": (True, 2), "outfielderBlocks": (True, 1.5),
        "clearances": (True, 1), "tacklesWonPercentage": (True, 2),
        # Aerial
        "aerialDuelsWonPercentage": (True, 2.5), "aerialDuelsWon": (True, 1.5),
        # On the ball
        "accuratePassesPercentage": (True, 2), "accurateOppositionHalfPasses": (True, 2),
        "accuratePasses": (True, 1.5), "accurateLongBalls": (True, 1.5),
        "possessionLostPercentage": (False, 1.5),          # ball security: losses per touch
        # Mistakes
        "dribbledPast": (False, 1.5), "errorLeadToShot": (False, 1.5), "errorLeadToGoal": (False, 2),
    },
    "Full-back": {
        "expectedAssists": (True, 2.5), "keyPasses": (True, 2), "accurateCrosses": (True, 2),
        "accurateCrossesPercentage": (True, 1.5), "accurateFinalThirdPasses": (True, 1.5),
        "groundDuelsWonPercentage": (True, 2), "tackles": (True, 1.5), "interceptions": (True, 1.5),
        "ballRecovery": (True, 1), "successfulDribbles": (True, 1),
        "dribbledPast": (False, 2), "errorLeadToShot": (False, 1),
    },
    "Central / defensive mid": {
        "accuratePassesPercentage": (True, 2.5), "accurateOppositionHalfPasses": (True, 2),
        "accurateFinalThirdPasses": (True, 2), "tacklesPlusInterceptions": (True, 2.5),
        "ballRecovery": (True, 2), "possessionWonAttThird": (True, 1.5),
        "groundDuelsWonPercentage": (True, 1.5), "accurateLongBalls": (True, 1),
        "keyPasses": (True, 1), "expectedAssists": (True, 1), "goals": (True, 1),
        "possessionLost": (False, 1.5), "dribbledPast": (False, 1),
    },
    "Winger / attacking mid": {
        "expectedAssists": (True, 3), "keyPasses": (True, 2.5), "bigChancesCreated": (True, 2.5),
        "successfulDribbles": (True, 2), "successfulDribblesPercentage": (True, 1.5),
        "nonPenaltyGoals": (True, 2.5), "nonPenaltyXG": (True, 2), "assists": (True, 2),
        "accurateFinalThirdPasses": (True, 1.5), "possessionWonAttThird": (True, 1),
        "wasFouled": (True, 1), "dispossessed": (False, 1),
    },
    "Striker": {
        "nonPenaltyGoals": (True, 3), "nonPenaltyXG": (True, 3), "goalsAboveXG": (True, 2),
        "goalConversionPercentage": (True, 2), "shotsOnTargetPercentage": (True, 1.5),
        "shotsOnTarget": (True, 1.5), "shotsFromInsideTheBox": (True, 1.5),
        "aerialDuelsWon": (True, 1), "assists": (True, 1.5), "expectedAssists": (True, 1),
        "keyPasses": (True, 1), "possessionWonAttThird": (True, 1),
    },
}

players with a detailed role from Transfermarkt: 322 of 326
role
Central / defensive mid    78
Winger / attacking mid     68
Centre-back                63
Full-back                  57
Striker                    36
Goalkeeper                 24


## 4. Score each stat against players in the same role (z-scores)

A z-score says how many "steps" above or below the role average a player is. Unlike percentiles, it keeps **how far ahead** someone is. The performance score is a **weighted average** of those z-scores, so a striker's goals count far more than his passing, and a holding midfielder's tackles and passing count far more than his goals. Scores are calculated across all 900+ minute players, not just the matched ones.

**Fairness adjustments:** penalty goals and penalty xG are removed for attackers, ball security is measured per touch for centre-backs, and scores are shrunk toward 0 for players with fewer minutes (a strong half-season is less certain than a strong full season).

In [5]:
def per90(frame, col):
    if "Percentage" in col:                      # percentages stay as they are
        return frame[col]
    return frame[col] / frame["minutesPlayed"] * 90

z_feats = pd.DataFrame(index=perf.index)
num = pd.Series(0.0, index=perf.index)           # weighted sum of z-scores
den = pd.Series(0.0, index=perf.index)           # sum of weights

for role, stats in role_stats.items():
    mask = perf["role"] == role
    for stat, (higher_better, weight) in stats.items():
        if stat not in perf.columns:
            print("skipping", stat)
            continue
        vals = per90(perf, stat).fillna(0)[mask]
        sd = vals.std()
        if not sd or np.isnan(sd):
            continue
        z = ((vals - vals.mean()) / sd)
        z = (z if higher_better else -z).clip(-3, 4)
        z_feats.loc[mask, f"{stat}_z"] = z
        num.loc[mask] += z * weight
        den.loc[mask] += weight

perf["perf_score"] = (num / den).fillna(0)   # 0 = average if a role has too few players to score
# Reliability: fewer minutes = noisier per-90 stats, so pull scores toward average (0)
SHRINK = 500                       # bigger = harsher on low minutes
perf["perf_score"] = perf["perf_score"] * perf["minutesPlayed"] / (perf["minutesPlayed"] + SHRINK)
w["perf_score"] = perf.loc[w.index, "perf_score"]

print(z_feats.shape[1], "stat features")
perf.sort_values("perf_score", ascending=False).groupby("role").head(3)[
    ["player_name", "team_name", "role", "perf_score"]].round(2)

48 stat features


,player_name,team_name,role,perf_score
29,Rayan Cherki,Manchester City,Winger / attacking mid,1.55
50,Bruno Fernandes,Manchester United,Winger / attacking mid,1.30
26,Jérémy Doku,Manchester City,Winger / attacking mid,1.17
27,Erling Haaland,Manchester City,Striker,1.15
24,Rodri,Manchester City,Central / defensive mid,1.02
88,Curtis Jones,Liverpool FC,Central / defensive mid,0.82
207,Moisés Caicedo,Chelsea,Central / defensive mid,0.71
137,Eli Junior Kroupi,Bournemouth,Striker,0.69
65,Benjamin Šeško,Manchester United,Striker,0.68
206,Reece James,Chelsea,Full-back,0.67


## 5. Expected wage

Only regular starters (1,500+ minutes) are judged. Within each role, a player whose performance is **X steps above his role's average performance** is expected to earn **X steps above his role's average wage** (on a log scale, so steps are % differences).

`PERF_WEIGHT = 0.5` keeps it conservative: one strong season moves pay only halfway, since a single season isn't proof.

- `expected_wage` = what his performance suggests he should earn
- `pay_ratio` = actual ÷ expected (2.0 = paid double, 0.5 = paid half)

In [6]:
# Expected wage: players further above their role's average performance
# get paid further above their role's average wage (on a log scale, so it's about % differences)
w = w[w["minutesPlayed"] >= 1500].copy()          # substantial minutes only

PERF_WEIGHT = 0.5      # how strongly performance moves pay: 1.0 = fully, 0.5 = more conservative

w["log_wage"] = np.log(w["annual_wage"])
by_role = w.groupby("role")
perf_z = (w["perf_score"] - by_role["perf_score"].transform("mean")) / by_role["perf_score"].transform("std")
w["expected_wage"] = np.exp(by_role["log_wage"].transform("mean")
                            + PERF_WEIGHT * perf_z * by_role["log_wage"].transform("std"))

w["wage_gap"] = w["annual_wage"] - w["expected_wage"]       # + = paid more than his performance earns
w["pay_ratio"] = w["annual_wage"] / w["expected_wage"]      # 2.0 = paid double, 0.5 = paid half
yw = np.log1p(w["annual_wage"])                              # used in section 8

print(f"{len(w)} players | correlation between performance score and wage: "
      f"{w['perf_score'].corr(yw):.2f}\n")
print("Players per role:")
print(w["role"].value_counts().to_string())

def ladder(name):
    role = w.loc[w["player_name"] == name, "role"].iloc[0]
    return (w[w["role"] == role].sort_values("perf_score", ascending=False)
            [["player_name", "team_name", "perf_score", "annual_wage", "expected_wage"]].round(2))

ladder("Hugo Bueno")

148 players | correlation between performance score and wage: 0.29

Players per role:
role
Central / defensive mid    33
Winger / attacking mid     33
Centre-back                32
Full-back                  28
Goalkeeper                 13
Striker                     9


,player_name,team_name,perf_score,annual_wage,expected_wage
206,Reece James,Chelsea,0.67,15876279,8494115.07
509,Hugo Bueno,Wolverhampton,0.62,680720,7982482.59
314,Lewis Hall,Newcastle United,0.56,444536,7402122.96
113,Lucas Digne,Aston Villa,0.38,7620614,5861688.87
415,Pedro Porro,Tottenham Hotspur,0.34,5397935,5599360.94
37,Matheus Nunes,Manchester City,0.32,8255665,5480261.80
387,Neco Williams,Nottingham Forest,0.31,3175256,5387863.50
259,Kenny Tete,Fulham,0.30,3175256,5344319.98
225,Malo Gusto,Chelsea,0.26,2857730,5073735.10
223,Marc Cucurella,Chelsea,0.22,11113395,4805556.44


## 6. Overpaid and underpaid

- **Underpaid:** top 20% of performers, paid at least €1M below expected
- **Overpaid:** paid at least €1M above expected
- Both lists are ranked by `pay_ratio`, so a player paid 3x his expected wage ranks above one paid 2x, whatever the size of the contract

In [7]:
top_perf = w["perf_score"].quantile(0.80)
over_mask  = (w["wage_gap"] > 1e6)  & (w["minutesPlayed"] >= 1500)
under_mask = (w["wage_gap"] < -1e6) & (w["perf_score"] >= top_perf)
w["verdict"] = np.select([over_mask, under_mask], ["Overpaid", "Underpaid"], "Fair / unclear")

w["overpaid_rank"]  = w["pay_ratio"].where(over_mask).rank(ascending=False, method="min")  # highest ratio = most overpaid
w["underpaid_rank"] = w["pay_ratio"].where(under_mask).rank(method="min")                  # lowest ratio = most underpaid

cols = ["player_name", "team_name", "role", "perf_score", "minutesPlayed",
        "annual_wage", "expected_wage", "wage_gap", "pay_ratio"]

print(f"Top-20% performance score cut-off: {top_perf:.2f}\n")
print(f"MOST OVERPAID ({over_mask.sum()} players)")
display(w[over_mask].sort_values("overpaid_rank")[["overpaid_rank"] + cols].head(15).round(2))

print(f"MOST UNDERPAID ({under_mask.sum()} players)")
display(w[under_mask].sort_values("underpaid_rank")[["underpaid_rank"] + cols].head(15).round(2))

print(w["verdict"].value_counts().to_string())

Top-20% performance score cut-off: 0.30

MOST OVERPAID (67 players)


,overpaid_rank,player_name,team_name,role,perf_score,minutesPlayed,annual_wage,expected_wage,wage_gap,pay_ratio
51,1.0,Casemiro,Manchester United,Central / defensive mid,0.09,2512,22226790,4395227.28,17831562.72,5.06
450,2.0,Tomáš Souček,West Ham United,Central / defensive mid,-0.76,1972,5715460,1190022.87,4525437.13,4.80
86,3.0,Mohamed Salah,Liverpool FC,Winger / attacking mid,0.28,2058,22226790,5892621.72,16334168.28,3.77
67,4.0,Luke Shaw,Manchester United,Full-back,-0.22,2969,9525767,2730252.98,6795514.02,3.49
12,5.0,Riccardo Calafiori,Arsenal,Full-back,-0.34,1545,7620614,2353074.03,5267539.97,3.24
315,6.0,Joelinton,Newcastle United,Central / defensive mid,-0.13,1863,9525767,3130043.70,6395723.30,3.04
42,7.0,Bernardo Silva,Manchester City,Central / defensive mid,0.35,2606,19051534,6519087.79,12532446.21,2.92
89,8.0,Alexis Mac Allister,Liverpool FC,Central / defensive mid,-0.08,2405,9525767,3342197.58,6183569.42,2.85
444,9.0,Lucas Paquetá,West Ham United,Winger / attacking mid,-0.29,1521,9525767,3490035.62,6035731.38,2.73
105,10.0,John McGinn,Aston Villa,Central / defensive mid,-0.16,1979,7620614,2969854.67,4650759.33,2.57


MOST UNDERPAID (14 players)


,underpaid_rank,player_name,team_name,role,perf_score,minutesPlayed,annual_wage,expected_wage,wage_gap,pay_ratio
314,1.0,Lewis Hall,Newcastle United,Full-back,0.56,1909,444536,7402122.96,-6957586.96,0.06
88,2.0,Curtis Jones,Liverpool FC,Central / defensive mid,0.82,1680,952577,13290260.71,-12337683.71,0.07
509,3.0,Hugo Bueno,Wolverhampton,Full-back,0.62,2304,680720,7982482.59,-7301762.59,0.09
26,4.0,Jérémy Doku,Manchester City,Winger / attacking mid,1.17,1574,3175256,13370874.67,-10195618.67,0.24
382,5.0,Elliot Anderson,Nottingham Forest,Central / defensive mid,0.60,3089,2540205,9545470.78,-7005265.78,0.27
233,6.0,James Garner,Everton,Central / defensive mid,0.35,3144,1905153,6538711.46,-4633558.46,0.29
213,7.0,Robert Sánchez,Chelsea,Goalkeeper,0.33,2861,3810307,7974140.23,-4163833.23,0.48
158,8.0,Igor Thiago,Brentford,Striker,0.40,3014,3175256,6599431.75,-3424175.75,0.48
194,9.0,Danny Welbeck,Brighton & Hove Albion,Striker,0.41,2068,3810307,6625566.51,-2815259.51,0.58
387,10.0,Neco Williams,Nottingham Forest,Full-back,0.31,2936,3175256,5387863.50,-2212607.50,0.59


verdict
Overpaid          67
Fair / unclear    67
Underpaid         14


## 7. Compare a player with others at a similar performance level

Change the name to look up anyone. Shows the **closest players by performance score** (any position), what they're actually paid, and what the model says they should earn. If the closest players are far away in score, that player is an outlier in his role.

In [8]:
def similar_players(name, n=8):
    me = w.loc[w["player_name"] == name].iloc[0]
    pool = w[w["player_name"] != name]
    pool = pool.assign(score_diff=(pool["perf_score"] - me["perf_score"]).round(2))
    closest = pool.reindex(pool["score_diff"].abs().sort_values().index).head(n)

    print(f"{name}: {me['role']}, perf score {me['perf_score']:.2f}, "
          f"paid {me['annual_wage']/1e6:.1f}M, expected {me['expected_wage']/1e6:.1f}M")
    print(f"{n} players with the closest performance score (any position): "
          f"scores {closest['perf_score'].min():.2f} to {closest['perf_score'].max():.2f}, "
          f"median wage {closest['annual_wage'].median()/1e6:.1f}M\n")
    table = pd.concat([w.loc[[me.name]].assign(score_diff=0.0), closest])
    return table[["player_name", "team_name", "role", "perf_score", "score_diff", "minutesPlayed",
                  "annual_wage", "expected_wage", "pay_ratio"]].round(2)

similar_players("Erling Haaland")

Erling Haaland: Striker, perf score 1.15, paid 23.8M, expected 10.5M
8 players with the closest performance score (any position): scores 0.60 to 1.30, median wage 7.6M



,player_name,team_name,role,perf_score,score_diff,minutesPlayed,annual_wage,expected_wage,pay_ratio
27,Erling Haaland,Manchester City,Striker,1.15,0.00,2778,23814418,10491642.91,2.27
26,Jérémy Doku,Manchester City,Winger / attacking mid,1.17,0.02,1574,3175256,13370874.67,0.24
50,Bruno Fernandes,Manchester United,Winger / attacking mid,1.30,0.15,2796,19051534,15047638.73,1.27
2,Bukayo Saka,Arsenal,Winger / attacking mid,0.91,-0.24,2055,12383497,10543232.92,1.17
88,Curtis Jones,Liverpool FC,Central / defensive mid,0.82,-0.33,1680,952577,13290260.71,0.07
207,Moisés Caicedo,Chelsea,Central / defensive mid,0.71,-0.44,2537,9525767,11341421.93,0.84
206,Reece James,Chelsea,Full-back,0.67,-0.48,1896,15876279,8494115.07,1.87
509,Hugo Bueno,Wolverhampton,Full-back,0.62,-0.53,2304,680720,7982482.59,0.09
418,Richarlison,Tottenham Hotspur,Striker,0.60,-0.55,1704,5715460,7473245.61,0.76


## 8. Which stats are linked to higher wages?

Not a second model: just the correlation between each stat (vs role average) and wage, inside each role. Positive = players who are better at this stat tend to earn more.

In [9]:
corr_rows = []
for col in z_feats.columns:
    vals = z_feats.loc[w.index, col]
    ok = vals.notna()
    if ok.sum() >= 15:
        corr_rows.append({"Stat": col[:-2], "Players": int(ok.sum()),
                          "Link to wage": round(vals[ok].corr(yw[ok]), 3)})
drivers = pd.DataFrame(corr_rows).groupby("Stat", as_index=False).agg(
    {"Link to wage": "mean", "Players": "sum"}).sort_values("Link to wage", ascending=False)
drivers.head(10)

,Stat,Link to wage,Players
26,possessionLostPercentage,0.602,32
5,accuratePasses,0.536,32
4,accurateOppositionHalfPasses,0.424,65
9,assists,0.404,42
8,aerialDuelsWonPercentage,0.382,32
2,accurateFinalThirdPasses,0.379,94
6,accuratePassesPercentage,0.295,78
21,keyPasses,0.288,103
18,goals,0.282,33
11,bigChancesCreated,0.281,33


## 9. Export for Tableau

Run this **last**. Creates, in `data/output/`:
- `tableau_players.csv`: one row per player, with performance, wages, verdicts and ranks
- `tableau_wage_drivers.csv`: which stats are linked to higher wages

In [10]:
w["perf_rank"] = w["perf_score"].rank(ascending=False, method="min").astype(int)
w["wage_rank"] = w["annual_wage"].rank(ascending=False, method="min").astype(int)
w["ratio_verdict"] = np.select([w["pay_ratio"] >= 2, w["pay_ratio"] <= 0.5],
                               ["Paid 2x+ expected", "Paid half or less"], "Within 2x")
w["availability"] = np.where(w["minutesPlayed"] < 1500, "Under 1,500 min", "Regular (1,500+ min)")
for col in ["goals", "assists", "expectedGoals", "expectedAssists", "keyPasses", "tackles", "interceptions"]:
    w[f"{col}_per90"] = (w[col].fillna(0) / w["minutesPlayed"] * 90).round(2)

export_cols = {
    "player_name": "Player", "team_name": "Club", "role": "Role",
    "age_2024_25": "Age (2024-25)", "appearances": "Appearances", "matchesStarted": "Starts",
    "minutesPlayed": "Minutes", "availability": "Availability", "rating": "Avg Rating",
    "goals": "Goals", "assists": "Assists",
    "goals_per90": "Goals per 90", "assists_per90": "Assists per 90",
    "expectedGoals_per90": "xG per 90", "expectedAssists_per90": "xA per 90",
    "keyPasses_per90": "Key Passes per 90", "tackles_per90": "Tackles per 90",
    "interceptions_per90": "Interceptions per 90",
    "perf_score": "Performance Score", "perf_rank": "Performance Rank",
    "annual_wage": "Annual Wage", "wage_rank": "Wage Rank",
    "expected_wage": "Expected Wage", "wage_gap": "Wage Gap", "pay_ratio": "Pay Ratio",
    "verdict": "Verdict", "ratio_verdict": "Ratio Verdict",
    "overpaid_rank": "Overpaid Rank", "underpaid_rank": "Underpaid Rank",
}
tableau = w[list(export_cols)].rename(columns=export_cols).copy()
tableau[["Annual Wage", "Expected Wage", "Wage Gap"]] = tableau[["Annual Wage", "Expected Wage", "Wage Gap"]].round(0)
tableau[["Performance Score", "Pay Ratio"]] = tableau[["Performance Score", "Pay Ratio"]].round(2)
tableau = tableau.sort_values("Pay Ratio", ascending=False)      # matches the ratio-based ranking

players_file = f"{OUT_DIR}/tableau_players.csv"
drivers_file = f"{OUT_DIR}/tableau_wage_drivers.csv"
tableau.to_csv(players_file, index=False, encoding="utf-8-sig")
drivers.to_csv(drivers_file, index=False, encoding="utf-8-sig")

print(f"{players_file}: {len(tableau)} players, {tableau.shape[1]} columns")
print(f"{drivers_file}: {len(drivers)} stats")
tableau.head(10)

data/output/tableau_players.csv: 148 players, 29 columns
data/output/tableau_wage_drivers.csv: 34 stats


,Player,Club,Role,Age (2024-25),Appearances,Starts,Minutes,Availability,Avg Rating,Goals,...,Performance Rank,Annual Wage,Wage Rank,Expected Wage,Wage Gap,Pay Ratio,Verdict,Ratio Verdict,Overpaid Rank,Underpaid Rank
51,Casemiro,Manchester United,Central / defensive mid,32,33,32,2512,"Regular (1,500+ min)",7.224242,9,...,55,22226790,2,4395227.0,17831563.0,5.06,Overpaid,Paid 2x+ expected,1.0,NaN
450,Tomáš Souček,West Ham United,Central / defensive mid,29,32,21,1972,"Regular (1,500+ min)",6.728125,5,...,148,5715460,54,1190023.0,4525437.0,4.80,Overpaid,Paid 2x+ expected,2.0,NaN
86,Mohamed Salah,Liverpool FC,Winger / attacking mid,32,25,22,2058,"Regular (1,500+ min)",6.876000,7,...,34,22226790,2,5892622.0,16334168.0,3.77,Overpaid,Paid 2x+ expected,3.0,NaN
67,Luke Shaw,Manchester United,Full-back,29,35,35,2969,"Regular (1,500+ min)",6.671429,0,...,120,9525767,18,2730253.0,6795514.0,3.49,Overpaid,Paid 2x+ expected,4.0,NaN
12,Riccardo Calafiori,Arsenal,Full-back,22,23,19,1545,"Regular (1,500+ min)",6.877273,1,...,135,7620614,35,2353074.0,5267540.0,3.24,Overpaid,Paid 2x+ expected,5.0,NaN
315,Joelinton,Newcastle United,Central / defensive mid,27,26,22,1863,"Regular (1,500+ min)",6.792308,2,...,104,9525767,18,3130044.0,6395723.0,3.04,Overpaid,Paid 2x+ expected,6.0,NaN
42,Bernardo Silva,Manchester City,Central / defensive mid,29,34,30,2606,"Regular (1,500+ min)",6.844118,2,...,22,19051534,4,6519088.0,12532446.0,2.92,Overpaid,Paid 2x+ expected,7.0,NaN
89,Alexis Mac Allister,Liverpool FC,Central / defensive mid,25,34,28,2405,"Regular (1,500+ min)",6.741176,2,...,99,9525767,18,3342198.0,6183569.0,2.85,Overpaid,Paid 2x+ expected,8.0,NaN
444,Lucas Paquetá,West Ham United,Winger / attacking mid,26,18,18,1521,"Regular (1,500+ min)",6.905556,4,...,128,9525767,18,3490036.0,6035731.0,2.73,Overpaid,Paid 2x+ expected,9.0,NaN
105,John McGinn,Aston Villa,Central / defensive mid,29,27,26,1979,"Regular (1,500+ min)",6.900000,4,...,111,7620614,35,2969855.0,4650759.0,2.57,Overpaid,Paid 2x+ expected,10.0,NaN
